# 00. 데이터 병합 (Data Merge)

서울시 상권분석 서비스(서울 열린데이터광장)에서 수집한 **8개 원본 데이터**를 자치구·업종·분기 기준으로 병합하여 하나의 분석용 통합 데이터셋을 만듭니다.

## 원본 데이터 (data/raw)

| 파일 | 내용 | 관측 단위 |
|------|------|-----------|
| 점포_자치구.csv | 점포수, 개업/폐업률, 프랜차이즈 | 분기 x 자치구 x 업종 |
| 매출_자치구.csv | 요일/시간/성별/연령대별 매출 | 분기 x 자치구 x 업종 |
| 상권변화_자치구.csv | 상권변화지표, 영업 개월 | 분기 x 자치구 |
| 유동인구_자치구.csv | 시간/요일/성별/연령대별 유동인구 | 분기 x 자치구 |
| 소득소비_자치구.csv | 소득, 항목별 지출 | 분기 x 자치구 |
| 임대료_자치구.csv | 전체 임대료 | 분기 x 자치구 |
| 상주인구_자치구.csv | 성별/연령대별 상주인구, 가구수 | 분기 x 자치구 |
| 직장인구_자치구.csv | 성별/연령대별 직장인구 | 분기 x 자치구 |

## 병합 전략
1. **업종 단위 병합**: 점포 + 매출 (`기준_년분기_코드`, `자치구_코드_명`, `서비스_업종_코드_명`)
2. **자치구 단위 병합**: 나머지 6개 (`기준_년분기_코드`, `자치구_코드_명`)
3. **최종 병합**: 업종 단위 데이터에 자치구 단위 데이터를 left join

→ 결과: **(39,975 행, 137 컬럼)**

In [1]:
import os
import re
import pandas as pd

RAW_DIR = '../data/raw'
OUTPUT_PATH = '../data/processed/merged_data.csv'

## 1. 원본 데이터 로드

8개 CSV 파일을 모두 불러옵니다. (서울 열린데이터광장 원본은 `utf-8-sig` 인코딩)

In [2]:
def load_raw(name):
    return pd.read_csv(os.path.join(RAW_DIR, name), encoding='utf-8-sig')

stores     = load_raw('점포_자치구.csv')      # 점포
sales      = load_raw('매출_자치구.csv')      # 매출
change     = load_raw('상권변화_자치구.csv')   # 상권변화
floating   = load_raw('유동인구_자치구.csv')   # 유동인구
income     = load_raw('소득소비_자치구.csv')   # 소득소비
rent       = load_raw('임대료_자치구.csv')     # 임대료
resident   = load_raw('상주인구_자치구.csv')   # 상주인구
working    = load_raw('직장인구_자치구.csv')   # 직장인구

for name, df in [('점포', stores), ('매출', sales), ('상권변화', change),
                 ('유동인구', floating), ('소득소비', income), ('임대료', rent),
                 ('상주인구', resident), ('직장인구', working)]:
    print(f'{name:6s}: {df.shape}')

점포    : (64715, 12)
매출    : (39975, 53)
상권변화  : (650, 9)
유동인구  : (650, 25)
소득소비  : (650, 16)
임대료   : (675, 3)
상주인구  : (650, 27)
직장인구  : (650, 24)


## 2. 병합 키 정리

**임대료 데이터는 `기준_년분기_코드`가 `'2019년 1분기'` 형태의 문자열**로 되어 있어, 다른 데이터의 숫자 코드(`20191`)와 병합되지 않습니다.  
→ `'2019년 1분기'` → `20191` 로 변환하고, 모든 데이터의 키 타입을 정수로 통일합니다.

In [3]:
def to_quarter_code(v):
    """'2019년 1분기' -> 20191, 이미 숫자 코드면 int로 변환."""
    s = str(v)
    m = re.match(r'\s*(\d{4})\D+([1-4])', s)
    if m:
        return int(m.group(1)) * 10 + int(m.group(2))
    return int(s)

# 임대료: 문자열 분기 표기 -> 숫자 코드
rent['기준_년분기_코드'] = rent['기준_년분기_코드'].map(to_quarter_code)

# 나머지: 정수 타입으로 통일
for df in [stores, sales, change, floating, income, resident, working]:
    df['기준_년분기_코드'] = df['기준_년분기_코드'].astype(int)

print('임대료 분기 코드 예시:', sorted(rent['기준_년분기_코드'].unique())[:5])

임대료 분기 코드 예시: [np.int64(20191), np.int64(20192), np.int64(20193), np.int64(20194), np.int64(20201)]


## 3. 업종 단위 데이터 병합 (점포 + 매출)

점포·매출 데이터는 `서비스_업종_코드_명` 단위까지 존재합니다.  
병합 시 중복되는 코드 컬럼(`자치구_코드`, `서비스_업종_코드`)과 매출의 파생 컬럼(`주중/주말`)은 제거합니다.  
(주중/주말 매출은 요일별 매출로 복원 가능한 중복 정보)

In [4]:
KEY_BIZ = ['기준_년분기_코드', '자치구_코드_명', '서비스_업종_코드_명']

# 중복 코드 컬럼 제거 (코드_명만 유지)
drop_codes = ['자치구_코드', '서비스_업종_코드']
stores_c = stores.drop(columns=[c for c in drop_codes if c in stores.columns])
sales_c  = sales.drop(columns=[c for c in drop_codes if c in sales.columns])

# 매출 파생(주중/주말) 컬럼 제거
sales_c = sales_c.drop(columns=[c for c in ['주중_매출_금액', '주말_매출_금액',
                                            '주중_매출_건수', '주말_매출_건수']
                                if c in sales_c.columns])

service_df = pd.merge(stores_c, sales_c, on=KEY_BIZ, how='outer')
print('업종 단위 병합 결과:', service_df.shape)

업종 단위 병합 결과: (64715, 54)


## 4. 자치구 단위 데이터 병합 (나머지 6개)

상권변화·유동인구·소득소비·임대료·상주인구·직장인구는 자치구 단위 데이터입니다.  
코드/명 중복 컬럼과 모델에 불필요한 세부 코드 컬럼을 정리하며 순차 병합합니다.
- `상권_변화_지표_명` 제거 (`상권_변화_지표` 코드 유지)
- `소득_구간_코드` 제거 (연속형 소득/지출 금액 유지)
- `아파트_가구_수`, `비_아파트_가구_수` 제거 (`총_가구_수` 유지)

In [5]:
KEY_GU = ['기준_년분기_코드', '자치구_코드_명']

change_c   = change.drop(columns=[c for c in ['자치구_코드', '상권_변화_지표_명']
                                  if c in change.columns])
floating_c = floating.drop(columns=[c for c in ['자치구_코드'] if c in floating.columns])
income_c   = income.drop(columns=[c for c in ['자치구_코드', '소득_구간_코드']
                                  if c in income.columns])
resident_c = resident.drop(columns=[c for c in ['자치구_코드', '아파트_가구_수', '비_아파트_가구_수']
                                    if c in resident.columns])
working_c  = working.drop(columns=[c for c in ['자치구_코드'] if c in working.columns])
# 임대료는 자치구_코드가 없음

district_df = change_c
for df in [floating_c, income_c, rent, resident_c, working_c]:
    district_df = pd.merge(district_df, df, on=KEY_GU, how='outer')

print('자치구 단위 병합 결과:', district_df.shape)

자치구 단위 병합 결과: (675, 85)


## 5. 최종 병합 및 저장

업종 단위 데이터에 자치구 단위 데이터를 left join 합니다.  
임대료 데이터에는 `'서울시 전체'` 행이 포함되어 있으나, 업종 단위 데이터에는 25개 자치구만 존재하므로 자연스럽게 걸러집니다.  
결측 행(일부 업종의 매출 누락 등)을 제거하면 최종 **(39,975 행, 137 컬럼)** 이 됩니다.

In [6]:
merged = pd.merge(service_df, district_df, on=KEY_GU, how='left')

# 결측 행 제거 (매출이 없는 업종 등)
merged = merged.dropna().reset_index(drop=True)

print('최종 병합 데이터:', merged.shape)
print('기간:', merged['기준_년분기_코드'].min(), '~', merged['기준_년분기_코드'].max())
print('자치구 수:', merged['자치구_코드_명'].nunique())
print('업종 수:', merged['서비스_업종_코드_명'].nunique())

최종 병합 데이터: (39975, 137)
기간: 20191 ~ 20252
자치구 수: 25
업종 수: 63


In [7]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 39975 entries, 0 to 39974
Columns: 137 entries, 기준_년분기_코드 to 여성연령대_60_이상_직장_인구_수
dtypes: float64(132), int64(2), str(3)
memory usage: 41.8 MB


In [8]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
merged.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT_PATH}')
print(f'  shape: {merged.shape}')

저장 완료: ../data/processed/merged_data.csv
  shape: (39975, 137)


## 정리

| 단계 | 처리 | 결과 |
|------|------|------|
| 로드 | 8개 원본 CSV | - |
| 키 정리 | 임대료 분기 표기 변환, 키 타입 통일 | - |
| 업종 병합 | 점포 + 매출 | (64,715, 54) |
| 자치구 병합 | 나머지 6개 | (675, 85) |
| 최종 병합 + 결측 제거 | left join | **(39,975, 137)** |

→ 다음: `01_EDA.ipynb`에서 병합된 데이터를 탐색합니다.